In [ ]:
import phoebe
import sys
sys.path.insert(1, '../scripts')
import phoebe_to_eBEER
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from scipy.optimize import curve_fit

import itertools
import time
import pickle

# Testing

In [ ]:
eccs1 = (0, 0.3, 0.8,)
incls1 = (0, 30, 60, 90, 120, 150, 180,)
per0s1 = (0, 30, 90, 150, 180, 240,)
qs1 = (1, 0.5, 2,)
ps1 = (1, 5, 10, 15, 20,)

Refl_ON = False

In [ ]:
start = time.time()

b_ = phoebe.default_binary()
b_.add_dataset('lc', times=0, label= 'lc01')
b_.set_value_all('gravb_bol', 0.32)
b_.set_value_all('ld_mode*', 'manual')
b_.set_value_all('ld_func*', 'linear')
b_.set_value_all('ld_coeffs*', [0.5])
b_.set_value_all('ntriangles', 100000)
b_['eclipse_method'] = 'only_horizon'
b_['passband'] = 'Kepler:mean'
b_.set_value_all('atm', 'phoenix')
b_.flip_constraint('mass@primary', solve_for='sma')
if not Refl_ON:
    b_.set_value('irrad_method', 'none')


b_2 = phoebe.default_binary()
b_2.add_dataset('lc', times=0, label= 'lc01')
b_2.set_value_all('gravb_bol', 0.32)
b_2.set_value_all('ld_mode*', 'manual')
b_2.set_value_all('ld_func*', 'linear')
b_2.set_value_all('ld_coeffs*', [0.5])
b_2.set_value_all('ntriangles', 10000)
b_2['eclipse_method'] = 'only_horizon'
b_2['passband'] = 'Kepler:mean'
b_2.set_value_all('atm', 'phoenix')
b_2.flip_constraint('mass@primary', solve_for='sma')
b_2.set_value('irrad_method', 'none')
b_2.set_value_all('distortion_method', value='sphere')
### SETUP DONE ###


def fit_scaling(arefl1, arefl2):
    scaled = np.zeros(2)
    scaled[0] = arefl1 * 7
    scaled[1] = arefl2 * 7
    return scaled

def fit_eBEER(t, arefl1, arefl2):
    scaled = fit_scaling(arefl1, arefl2)
    flux = phoebe_to_eBEER.get_eBEER_lc(b_, t, 0, 0, scaled[0], scaled[1])
    return flux


### Start Loop ###
combinations = itertools.product(eccs, incls, per0s, ps, qs)
comb_list = list(combinations)

periods = np.zeros(len(comb_list))
PHOEBE_A = np.zeros(len(comb_list))
eBEER_A = np.zeros(len(comb_list))
RMSDs = np.zeros(len(comb_list))

for i, comb in enumerate(comb_list):
    start_sys = time.time()
    ecc, incl, per0, p, q = comb
    comb_str = f'lc_{p}_{int(ecc*10)}_{incl}_{per0}_{int(q*10)}'
    print('System:', comb_str)

    times = np.linspace(0,p,101)
    b_['lc01@dataset@times']= times
    b_['orbit@period'].set_value(p * u.day)
    b_['orbit@incl'].set_value(incl * u.deg)
    b_['orbit@ecc'].set_value(ecc)
    b_['orbit@per0'].set_value(per0 * u.deg)
    b_['mass@primary@component'].set_value(1 * u.M_sun)
    b_['orbit@q'].set_value(q)
    b_['primary']['requiv'].set_value(1 * u.R_sun)
    b_['secondary']['requiv'].set_value(q**0.8 * u.R_sun)
    setup_done = time.time()
    print('\tSystem Setup Duration:', setup_done - start_sys)
    # print_pb_params(b_)
    try:
        b_.run_compute(model= comb_str, overwrite= True)
        lc_done = time.time()
        print('\tLC Dataset Duration:', lc_done - setup_done)
    except:
        print('\tFAILED COMPUTE')
        continue
    
    b_2['orbit@period'].set_value(p * u.day)
    b_2['orbit@incl'].set_value(incl * u.deg)
    b_2['orbit@ecc'].set_value(ecc)
    b_2['orbit@per0'].set_value(per0 * u.deg)
    b_2['mass@primary@component'].set_value(1 * u.M_sun)
    b_2['orbit@q'].set_value(q)
    b_2['primary']['requiv'].set_value(1 * u.R_sun)
    b_2['secondary']['requiv'].set_value(q**0.8 * u.R_sun)
    # print_pb_params(b_2)
    b_2.run_compute(overwrite= True)

    flux = b_[f'fluxes@lc01@{comb_str}@model'].value
    ref_flux = b_2['fluxes@lc01@latest@model'].value
    relative_flux_ = (flux - ref_flux)/ref_flux * 1e6

    if not Refl_ON:
        eBEER_flux = phoebe_to_eBEER.get_eBEER_lc(b_, times, 0, 0, 0, 0)
    else:
        popt_, pcov = curve_fit(fit_eBEER, times, relative_flux_, p0= (0.5, 0.5), bounds= ([0, 0], [1, 1]))
        rescaled = fit_scaling(popt_[0], popt_[1])
        fit_flux_ = fit_eBEER(times, popt_[0], popt_[1])
        print('\tFit Duration:', time.time() - lc_done)
        print('\t\t', rescaled)

    if not Refl_ON: residuals = (relative_flux_ - eBEER_flux)
    else: residuals = (relative_flux_ - fit_flux_)
    
    MSD = np.mean(residuals**2)
    RMSD = np.sqrt(MSD)
    RMSDs[i] = RMSD
    periods[i] = p

    if not Refl_ON:
        eBEER_A[i] = (np.max(eBEER_flux) - np.min(eBEER_flux)) / 2
    else:
        eBEER_A[i] = (np.max(fit_flux_) - np.min(fit_flux_)) / 2
    PHOEBE_A[i] = (np.max(relative_flux_) - np.min(relative_flux_)) / 2
    

    print('\tTotal Duration:', time.time() - start_sys)

    t0 = b_['orbit@component@t0_perpass'].get_value(u.day)
    phase = (times - t0) % p
    plt.figure(figsize=(8, 5))
    plt.plot(times, relative_flux_, 'b.', markersize=1, label= 'PHOEBE LC')
    if not Refl_ON:
        plt.plot(times, eBEER_flux, 'g', markersize=1, label= 'eBEER LC')
    else:
        plt.plot(phase, fit_flux_, 'r', markersize=1, label= 'eBEER Fit')
    plt.title(fr'{i} - ecc:{ecc}, incl:{incl}, $\omega$:{per0}, Porb:{p}, q:{q}')
    plt.xlabel('Time (days)')
    plt.ylabel('dFlux (ppm)')
    plt.legend()
    plt.show()

In [ ]:
plt.scatter(periods, RMSDs)
plt.xscale('log')
plt.xlabel('Period (days)')
plt.ylabel('RMSD (ppm)')
plt.title('sys_config')
plt.legend()

In [ ]:
plt.plot(periods, eBEER_A, 'b.', label= 'eBEER')
plt.plot(periods, PHOEBE_A, 'r.', label= 'PHOEBE')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Period (days)')
plt.ylabel('Amplitude (ppm)')
plt.title('LC Amplitude vs Orbital Period')
plt.legend()